重新考察全连接层

1.将输入和输出变形为矩阵（宽度，高度）

2.将权重变形为 4-D 张量 (h,w)->(h',w')

3.V是W的重新索引

# 从全连接层到卷积

## 1. 全连接处理图像的两个毛病
老办法：把图片拉平成一维向量（28×28 → 784）。
- 毛病一：空间结构丢了。上下相邻的像素，拉平后隔了一整行，网络要重新学"谁挨着谁"。
- 毛病二：参数爆炸。1000×1000 的图全连接到同尺寸输出要 (10⁶)² = 10¹² 个参数，训不动。

## 2. 第一步：输入和输出别拉平，保持成矩阵
让 X 和隐藏表示 H 都保持 [i,j] 的二维形式，把"位置"写进数据结构本身。

## 3. 第二步：把权重写成 4-D 张量
输出位置 (i,j) 要连到每个输入位置 (k,l)，每根连线一个权重：
    H[i,j] = U[i,j] + Σ_k Σ_l W[i,j,k,l]·X[k,l]
- (i,j) = 连到输出图的哪个位置
- (k,l) = 来自输入图的哪个位置
- 2 + 2 = 4，这就是"4-D 张量"的全部由来（一根线的两个端点各要 2 个坐标）
- U[i,j] = 偏置，每个输出位置一个

把这层关系摊成表（2×2 → 2×2）：
- 一行 = 一个输出位置的全部权重 → 决定一个输出像素 H[i,j]
- 一列 = 一个输入位置分发给所有输出的权重
- 16 个格子 = 16 个独立参数，跟 Linear(4,4) 完全一样

## 4. 关键：这步只是换记法，参数一个没少
写成 4-D 本身零收益（不省参数、不省计算），跟拉平做 Linear 一模一样。

那为什么要写？——为了能定义"距离"。
有了 W[i,j,k,l]，才能写出 i−k（垂直距离）和 j−l（水平距离）。
有了距离，才能施加两条原则：
- 局部性：只让距离近的相连
- 平移不变性：距离相同的一对共用权重
类比：找"3 公里内的城市"需要经纬度；只有编号（一维）就没法定义距离。

## 5. 第三步：4-D 怎么变成 2-D 卷积核（4 步）

### 第 1 步 换变量（k = i+a, l = j+b）
    H[i,j] = U[i,j] + Σ_a Σ_b W[i,j,i+a,j+b]·X[i+a,j+b]
凭什么：只是换"数法"。按绝对坐标遍历 vs 按相对偏移遍历，
遍历的是同一批格子，一个不多一个不少。

### 第 2 步 换名（纯记号）
    V[i,j,a,b] ≡ W[i,j,i+a,j+b]
    H[i,j] = U[i,j] + Σ_a Σ_b V[i,j,a,b]·X[i+a,j+b]
凭什么：纯换符号，参数没少。目的是让偏移 (a,b) 成为显式自变量，好对 (i,j) 下手。

### 第 3 步 平移不变性（人为假设，不是推导！）
    V[i,j,a,b] → V[a,b]
凭什么：先验假设——同一个特征检测器放在图的任何位置，参数应该一样。
        我们主动规定权重不许依赖绝对位置 (i,j)。
具体体现：W[1,1,0,0] 与 W[2,2,1,1] 的偏移都是 (−1,−1)，
        于是合并成同一个参数 V[−1,−1]。

### 第 4 步 局部性
    H[i,j] = U + Σ_{|a|≤Δ} Σ_{|b|≤Δ} V[a,b]·X[i+a,j+b]
凭什么：判断一个位置只看附近，远处像素没用 → 求和范围缩到 (2Δ+1)² 的小窗口。

## 6. 参数量阶梯（1000×1000 的图，Δ=1）
| 阶段 | 参数量 | 数值 |
|---|---|---|
| 起点：4-D 全连接 | (hw)² | 10¹² |
| 第 1、2 步（换变量、换名） | 没变 | 10¹² |
| 第 3 步 平移不变性 | hw（全图共用一套） | 10⁶ |
| 第 4 步 局部性 | (2Δ+1)² | 9 |

两步各砍一刀：平移不变性砍"每个位置各存一份"，局部性砍"每个位置要看全图"。
最终参数与图片尺寸彻底无关。

## 7. 那 9 个参数能反映整张图吗
- 参数存的是**规则/判据**，不是图像内容。
  9 个数（如 −1 0 1 / −1 0 1 / −1 0 1）= 一条判据：左暗右亮 = 竖边。
  它像一把 3×3 的尺子，本身不含任何图片信息。
- 图像信息存在**特征图（activation）**&#8203;里，不在权重里。
- 覆盖全图靠滑动复用：一个核逐格滑过全图，每滑一次算一次点积，
  得到一张与原图大小相当的特征图，每格 = 该处对该模式的响应强度。
  参数还是 9 个，却产出覆盖全图的响应。
- 一层有 K 个核 → K 张特征图（这就是"通道"的由来），参数 = K×9，
  认出竖边、横边、纹理等 K 种模式。真正"反映图像"的是这 K 张图的叠加。
- 单层只看 3×3，但多层堆叠感受野扩大：1 层 3×3 → 2 层 5×5 → 3 层 7×7。
  堆够层数，深层神经元就能"看到"整张图。

## 8. 参数少是优势，不是缺陷
- 权重共享 = 强归纳偏置：事先假设"同一特征出现在哪都一样"，
  网络不必在每个位置各学一份 → 泛化更好。
- 参数量与图片尺寸无关：100×100 和 1000×1000 的图都用 9 个参数。

## 9. 核心结论
- 保持二维：为了保住空间结构，别拉平。
- 4-D 是中间形态：把"谁连谁、相距多远"显式写出来，本身不省参数。
- 卷积 = 平移不变性 + 局部性，把 4-D 权重压成小小的 2-D 核。
- 第 3 步（去 i,j）是人为假设不是数学推导，它是卷积的定义本身。
- 覆盖全图靠"滑动复用 + 多通道 + 多层感受野"，不靠单核参数多。
- 注意：nn.Conv2d 的权重也是 4-D：(输出通道, 输入通道, 核高, 核宽)，
  那两维是通道，与推导里"输出位置×输入位置"的 4-D 含义不同。
